# ЛР 2. Метрики оцінювання моделей машинного навчання та їх вплив на вибір моделі

**Виконав Гуленко Назар, студент групи ІДС-501**

## Завдання 1. Вибір метрики у задачі класифікації

Банк розробляє модель для прогнозування ймовірності дефолту клієнтів за кредитами. Модель повинна передбачити, чи поверне клієнт отриманий кредит.  
У такій задачі можливі два типи помилок:

- False Negative - модель вважає клієнта надійним, але він не повертає кредит;  
- False Positive - модель відхиляє клієнта, який насправді міг би повернути кредит.  

Для виконання завдання використовуйте набір даних `Default of Credit Card Clients` (код для завантаження даних залишила під описом завдання).  
Цей набір даних містить інформацію про клієнтів банку та характеристики їхніх кредитів.  
Цільовою змінною є `class`, яка показує результат повернення кредиту:

- good - кредит було повернуто;  
- bad - клієнт не повернув кредит (дефолт).  

**1. Завантажте набір даних**

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.datasets import fetch_openml
# Завантаження спеціалізованого набору даних для кредитного скорингу
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
# Попередній огляд структури даних
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   checking_status         1000 non-null   category
 1   duration                1000 non-null   int64   
 2   credit_history          1000 non-null   category
 3   purpose                 1000 non-null   category
 4   credit_amount           1000 non-null   int64   
 5   savings_status          1000 non-null   category
 6   employment              1000 non-null   category
 7   installment_commitment  1000 non-null   int64   
 8   personal_status         1000 non-null   category
 9   other_parties           1000 non-null   category
 10  residence_since         1000 non-null   int64   
 11  property_magnitude      1000 non-null   category
 12  age                     1000 non-null   int64   
 13  other_payment_plans     1000 non-null   category
 14  housing                 1

**2.Виділіть ознаки та цільову змінну**

In [4]:
# Відокремлення цільової змінної від предикторів
X = df.drop("class", axis=1)
# Бінаризація цільової змінної: 0 - надійний клієнт, 1 - ризик дефолту
y = df["class"].map({"good": 0, "bad": 1})

In [5]:
# Трансформація категоріальних ознак у фіктивні змінні (one-hot encoding) 
# для коректної роботи лінійних алгоритмів
X = pd.get_dummies(X, drop_first=True)

**3.Розділіть дані на навчальну та тестову вибірки**

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

**4.Побудуйте модель класифікації (обираєте будь-який алгоритм)**

In [7]:
from sklearn.preprocessing import StandardScaler
#Стандартизація для логістичної регресії
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

**5-6. Проведіть чотири окремих запуски GridSearchCV, кожного разу використовуючи іншу метрику оцінювання:**

Для кожної метрики:
визначте найкращі параметри моделі;
побудуйте модель з цими параметрами;
обчисліть метрики на тестовій вибірці;
виведіть матрицю помилок та звіт про ккласифікацію.


   - `accuracy`  


In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Визначення простору гіперпараметрів для пошуку оптимальної конфігурації
param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "class_weight": [None, "balanced"]
}

# Оптимізація моделі за метрикою Accuracy (загальна частка правильних прогнозів)
grid_acc = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring="accuracy")
grid_acc.fit(X_train, y_train)

print("Найкращі параметри (accuracy):", grid_acc.best_params_)

# побудова моделі з найкращими параметрами
model_acc = grid_acc.best_estimator_
# прогноз на тестовій вибірці
y_pred_acc = model_acc.predict(X_test)
# матриця помилок
print(confusion_matrix(y_test, y_pred_acc))
# обчислення метрик
print(classification_report(y_test, y_pred_acc))

Найкращі параметри (accuracy): {'C': 0.1, 'class_weight': None}
[[117  23]
 [ 30  30]]
              precision    recall  f1-score   support

           0       0.80      0.84      0.82       140
           1       0.57      0.50      0.53        60

    accuracy                           0.73       200
   macro avg       0.68      0.67      0.67       200
weighted avg       0.73      0.73      0.73       200



**Accuracy (0.73)**

Оптимізація за загальною точністю демонструє високу ефективність на мажоритарному класі (надійні клієнти), де $F1\text{-score}$ сягає $0.82$. Проте в умовах дисбалансу класів (~$30\%$ дефолтів), дана метрика виявляється нерепрезентативною. Модель ідентифікує лише половину ризикових випадків ($Recall = 0.50$), що є критично низьким показником для банківського сектору, де ціна пропущеного дефолту значно перевищує ціну хибної відмови.

**Клас 0 (good, більшість)**

Модель добре працює з надійними клієнтами. З 140 випадків правильно класифіковано 117, що дає recall 0.84. Precision 0.80 означає, що більшість передбачень цього класу є коректними. F1-score 0.82 підтверджує стабільну якість для цього класу.

**Клас 1 (bad, меншість)**

Для дефолтних клієнтів результати значно гірші: з 60 випадків правильно визначено лише 30 (recall 0.50). Precision 0.57 свідчить про помітну кількість хибних спрацювань. F1-score 0.53 показує слабку здатність моделі працювати з ризиковими клієнтами. Модель часто пропускає дефолти, що є критичною проблемою.

   - `precision`  


In [10]:
# Регулювання моделі для максимізації Precision (мінімізація хибнопозитивних рішень)
grid_prec = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring="precision")
grid_prec.fit(X_train, y_train)

print("Best params (precision):", grid_prec.best_params_)

# побудова моделі з найкращими параметрами
model_prec = grid_prec.best_estimator_
# прогноз на тестовій вибірці
y_pred_prec = model_prec.predict(X_test)
# матриця помилок
print(confusion_matrix(y_test, y_pred_prec))
# обчислення метрик
print(classification_report(y_test, y_pred_prec))

Best params (precision): {'C': 0.01, 'class_weight': None}
[[127  13]
 [ 36  24]]
              precision    recall  f1-score   support

           0       0.78      0.91      0.84       140
           1       0.65      0.40      0.49        60

    accuracy                           0.76       200
   macro avg       0.71      0.65      0.67       200
weighted avg       0.74      0.76      0.74       200



**Precision (0.76)**

При фокусуванні на точності прогнозів ризику, модель стає консервативною. Показник $Recall$ для дефолтів падає до $0.40$, що свідчить про ігнорування більшості потенційно проблемних кредитів. Така модель максимізує впевненість у тому, що клієнт, позначений як "bad", дійсно є таким, проте її здатність до загального виявлення ризиків ($F1 = 0.49$) є незадовільною для практичного застосування.

**Клас 0 (good, більшість)**

Модель ще краще визначає надійних клієнтів. recall 0.91 означає, що майже всі такі випадки виявляються правильно. Precision 0.78 залишається на хорошому рівні, що забезпечує стабільний F1-score 0.84.

**Клас 1 (bad, меншість)**

Для дефолтів ситуація погіршується. Правильно виявлено лише 24 з 60 випадків (recall 0.40). Хоча precision підвищується до 0.65 (менше хибних відмов), модель пропускає більшість проблемних клієнтів. F1-score 0.49 підтверджує, що така модель неефективна для виявлення ризиків.

   - `recall`  

In [11]:
# Орієнтація на Recall (максимізація виявлення дефолтних клієнтів)
grid_rec = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring="recall")
grid_rec.fit(X_train, y_train)

print("Best params (recall):", grid_rec.best_params_)

# побудова моделі з найкращими параметрами
model_rec = grid_rec.best_estimator_
# прогноз на тестовій вибірці
y_pred_rec = model_rec.predict(X_test)
# матриця помилок
print(confusion_matrix(y_test, y_pred_rec))
# обчислення метрик
print(classification_report(y_test, y_pred_rec))

Best params (recall): {'C': 0.1, 'class_weight': 'balanced'}
[[97 43]
 [18 42]]
              precision    recall  f1-score   support

           0       0.84      0.69      0.76       140
           1       0.49      0.70      0.58        60

    accuracy                           0.69       200
   macro avg       0.67      0.70      0.67       200
weighted avg       0.74      0.69      0.71       200



**Recall (0.69)**

Використання class_weight: 'balanced' дозволило суттєво підвищити чутливість алгоритму до дефолтів. Модель коректно визначає $70\%$ ризикових клієнтів. Незважаючи на зниження загальної точності через зростання хибних тривог ($Precision$ для класу $1$ впав до $0.49$), цей підхід є більш доцільним з точки зору мінімізації фінансових втрат від невиплат за кредитами.

**Клас 0 (good, більшість)**

Якість для надійних клієнтів знизилась. recall 0.69 означає більше неправильних класифікацій. Precision 0.84 залишається високим, але F1-score падає до 0.76.

**Клас 1 (bad, меншість)**

Для дефолтів спостерігається суттєве покращення: правильно визначено 42 з 60 випадків (recall 0.70). Precision 0.49 означає, що майже половина передбачень є хибними. F1-score 0.58 показує кращу здатність виявляти дефолти, хоча ціною більшої кількості помилкових відмов.

   - `f1`  

In [12]:
grid_f1 = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring="f1")
grid_f1.fit(X_train, y_train)

print("Best params (f1):", grid_f1.best_params_)

model_f1 = grid_f1.best_estimator_
y_pred_f1 = model_f1.predict(X_test)

print(confusion_matrix(y_test, y_pred_f1))
print(classification_report(y_test, y_pred_f1))

Best params (f1): {'C': 1, 'class_weight': 'balanced'}
[[100  40]
 [ 17  43]]
              precision    recall  f1-score   support

           0       0.85      0.71      0.78       140
           1       0.52      0.72      0.60        60

    accuracy                           0.71       200
   macro avg       0.69      0.72      0.69       200
weighted avg       0.75      0.71      0.73       200



**F1-score (0.71)**

Оптимізація за $F1\text{-score}$ забезпечила найбільш стійкий компроміс. Модель демонструє високий рівень повноти ($Recall = 0.72$) при вищій точності ($0.52$) порівняно з попередньою ітерацією. Це призводить до кращої предиктивної здатності на обох класах одночасно, роблячи цю конфігурацію оптимальною для впровадження у систему прийняття рішень.

**Клас 0 (good, більшість)**

Показники залишаються достатньо високими. Показник recall 0.71 та precision 0.85 дають F1-score 0.78. Це означає, що модель все ще добре працює з надійними клієнтами.

**Клас 1 (bad, меншість)**

Для дефолтів досягнуто найкращий баланс. З 60 випадків правильно визначено 43 (recall 0.72), а precision 0.52 є кращим, ніж у моделі з оптимізацією лише за recall. F1-score 0.60 підтверджує більш стабільну якість.



**7.Порівняйте результати**

Моделі, оптимізовані за accuracy та precision, добре працюють з класом 0, але погано виявляють дефолти. Модель за recall значно краще знаходить проблемних клієнтів, але створює багато хибних спрацювань. Найкращим компромісом є модель, оптимізована за F1-score, оскільки вона забезпечує достатньо високу якість для обох класів і є найбільш придатною для задачі прогнозування кредитного ризику.

## Завдання 2

Агентство нерухомості розробляє модель для прогнозування вартості житла.
При оцінюванні якості моделі можуть використовуватися різні метрики:

MAE характеризує середню величину помилки прогнозу;
RMSE сильніше враховує великі помилки;
R² показує, яку частку варіації даних пояснює модель.
Вибір метрики може вплинути на те, яка модель буде визнана найкращою.
Для виконання завдання завантажте набір даних California Housing (код для завантаження залишила під описом завдання), який містить інформацію про характеристики житлових районів у Каліфорнії.

Цільовою змінною є MedHouseVal - медіанна вартість житла у відповідному районі.

**1. Завантажте набір даних**

In [13]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df1 = data.frame
df1.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [14]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


**2. Підготуйте дані**

Виділіть ознаки та цільову змінну

In [22]:
# Визначення цільового вектора (вартість житла) та матриці ознак
X = df1.drop("MedHouseVal", axis=1)
y = df1["MedHouseVal"]

# Поділ даних на вибірки для навчання та валідації
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

**3.Побудуйте модель регресії (алгоритм обираєте будь-який).**

In [23]:
from sklearn.ensemble import HistGradientBoostingRegressor
# Ініціалізація градієнтного бустингу на основі гістограм, що ефективно 
# обробляє великі масиви даних та нелінійні залежності.
model_gr = HistGradientBoostingRegressor(random_state=123)

**4-5.Проведіть три окремі запуски GridSearchCV, кожного разу використовуючи іншу метрику оцінювання:**

In [24]:
# Формування сітки гіперпараметрів для тонкого налаштування ансамблю
param_grid = {
    "max_depth": [2, 3, 5, None],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
    "max_iter": [50, 100, 200, 500],
    "min_samples_leaf": [1, 5, 10, 20, 50]
}

In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

scoring_list = [
    "neg_mean_absolute_error",
    "neg_root_mean_squared_error",
    "r2"
]

In [ ]:
# Паралелізований пошук найкращих параметрів для різних функцій втрат
for score in scoring_list:
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring=score,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("\n==============================")
    print("Скоринг:", score)
    print("Найкращі параметри:", grid.best_params_)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

**6.Порівняйте результати**

**Модель 1 (Оптимізація за MAE):**

Спрямована на мінімізацію середнього модуля помилки. Модель продемонструвала найкращу узагальнюючу здатність ($R^2 = 0.8528$), що вказує на її стійкість до викидів у цінах на нерухомість.

**Моделі 2 та 3 (Оптимізація за RMSE та R²):**

Обидві метрики призвели до ідентичних конфігурацій гіперпараметрів. RMSE більш суворо штрафує за значні відхилення, що важливо для уникнення грубих помилок в оцінці об'єктів.

**Порівняння результатів**

1. $MAE = 0.289$: Середня абсолютна помилка прогнозу складає ~$28,900.
   
2. $RMSE = 0.443$: Наявність більшого значення RMSE порівняно з MAE підтверджує присутність у вибірці специфічних районів ("аномалій"), де модель помиляється суттєвіше.

3. $R^2 = 0.85$: Модель пояснює $85\%$ дисперсії цільової змінної, що свідчить про високу якість підбору факторів впливу (локація, дохід, вік будинку) на ринкову вартість.

Для задачі оцінки нерухомості вибір метрики не призвів до кардинальних змін у структурі моделі, проте використання MAE як цільової функції дозволило досягти дещо вищої загальної точності на тестових даних.